# [BASELINE] supra50m → Mercury-2-style diffusion LM (original-paper recipe)

> **Note.** This notebook implements the *original Mercury/DiffuLLaMA path* — **gradual
> adaptation (continued training) of the transformer itself**. That is NOT the project's
> method: it serves as the **reference ceiling**. The project-method port — where model B
> is produced *exclusively by the weight-translator C* with zero training of B — is
> `c_mercury_port.ipynb` in this folder.

**Goal.** Translate the knowledge of `SupraLabs/Supra-50M-Instruct` (an autoregressive
12-layer Llama) into a **Mercury-like discrete-diffusion LM**: the same backbone acting as
a *denoiser*, generating by **parallel iterative denoising** instead of autoregression.

**Constraint honored (the project's thesis).** No inference on any external corpus: the
*only* data used is **sampled from the donor model itself** (the allowed 'Q-A dataset
captured from the model'). The weights themselves carry over directly — the 'translation'
is (1) causal → bidirectional attention (annealed), (2) AR objective → masked-diffusion
LLaDA ELBO, (3) AR decoding → confidence-based parallel denoising. This is the proven
DiffuLLaMA / Dream / LLaDA recipe, whose core was verified by 6 CPU unit tests in phase 7
(`experiments/diffusion_port/diffusion.py`).

**Setup.** Enable **GPU** + **Internet**, Run All. Downloads the model from HF, generates
its own training corpus, adapts, evaluates, and saves the translated checkpoint to
`/kaggle/working/supra_mercury2.safetensors`.

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import glob, json, math, time, torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors.torch import load_file, save_file

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)

MODEL_ID = 'SupraLabs/Supra-50M-Instruct'

def resolve_model():
    env = os.environ.get('CKPT_DIR')
    if env and os.path.exists(os.path.join(env, 'config.json')): return env
    ds = glob.glob('/kaggle/input/**/config.json', recursive=True)
    if ds: return os.path.dirname(ds[0])
    from huggingface_hub import snapshot_download
    return snapshot_download(MODEL_ID, allow_patterns=['config.json', 'model.safetensors',
                                                       'tokenizer.json'])

MODEL_DIR = resolve_model()

# --- budget knobs (one Kaggle GPU session) ---
N_GEN    = 1536      # self-generated training sequences (the whole data budget)
N_HELD   = 64        # held-out self-generated sequences
SEQ_LEN  = 256
BS       = 16        # fp32 full fine-tune fits a T4 comfortably at this size
STEPS    = 6000      # adaptation steps  (~25M tokens seen)
ANNEAL_FRAC = 0.5    # causal->bidirectional anneal over the first half
LR       = 1e-4
WARMUP   = 100
EPS_T    = 0.05      # mask-rate floor (keeps the 1/t ELBO weight sane)
print('device', DEV, '| model', MODEL_DIR)

In [ ]:
tj = json.load(open(os.path.join(MODEL_DIR, 'tokenizer.json')))
inv_vocab = {i: t for t, i in tj['model']['vocab'].items()}
def decode(ids):
    return ''.join(inv_vocab.get(int(i), '?') for i in ids).replace('\u2581', ' ') \
             .replace('\u0120', ' ').replace('\u010a', '\n')
print('vocab', len(inv_vocab))

In [ ]:
def rotate_half(x):
    d = x.shape[-1] // 2
    return torch.cat([-x[..., d:], x[..., :d]], dim=-1)

class DiffLlama(nn.Module):
    """supra50m backbone as a trainable denoiser: config-driven Llama (RoPE/GQA/SwiGLU,
    tied embeddings) loaded from safetensors, with (a) one extra embedding row = [MASK]
    (mean-init) and (b) per-sequence causal/bidirectional attention for annealing."""
    def __init__(self, path, dev=DEV):
        super().__init__()
        cfg = json.load(open(os.path.join(path, 'config.json')))
        self.H = cfg['num_attention_heads']
        self.KV = cfg.get('num_key_value_heads', self.H)
        self.d = cfg['hidden_size']
        self.hd = cfg.get('head_dim', self.d // self.H)
        self.L, self.eps = cfg['num_hidden_layers'], cfg.get('rms_norm_eps', 1e-5)
        rp = cfg.get('rope_parameters') or {}
        self.theta = cfg.get('rope_theta', rp.get('rope_theta', 10000))
        self.V = cfg['vocab_size']            # original vocab; MASK_ID == self.V
        sd = load_file(os.path.join(path, 'model.safetensors'))
        for k, v in sd.items():
            v = v.float()
            if k == 'model.embed_tokens.weight':         # append the [MASK] row (mean-init)
                v = torch.cat([v, v.mean(0, keepdim=True)], 0)
            setattr(self, k.replace('.', '_'), nn.Parameter(v))
        self.names = [k for k in sd]
        self.to(dev); self.dev = dev
    def g(self, k): return getattr(self, k.replace('.', '_'))
    def rms(self, x, w):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * w
    def rope(self, x, pos):
        inv = 1.0 / (self.theta ** (torch.arange(0, self.hd, 2, device=self.dev).float() / self.hd))
        fr = pos[:, None].float() * inv[None, :]
        cos = torch.cat([fr.cos(), fr.cos()], -1)[None, None]
        sin = torch.cat([fr.sin(), fr.sin()], -1)[None, None]
        return x * cos + rotate_half(x) * sin
    def forward(self, idx, causal_rows=None, causal=True):
        """causal_rows: (B,) bool — per-sequence causal flag (annealing mixture);
        else uniform `causal`. Bidirectional == no attention bias at all."""
        B, T = idx.shape
        pos = torch.arange(T, device=self.dev)
        x = self.g('model.embed_tokens.weight')[idx]
        tri = torch.full((T, T), float('-inf'), device=self.dev).triu(1)
        if causal_rows is not None:
            bias = torch.where(causal_rows.view(B, 1, 1, 1), tri, torch.zeros_like(tri))
        else:
            bias = tri[None, None] if causal else None
        for l in range(self.L):
            p = lambda n: self.g(f'model.layers.{l}.{n}.weight')
            h = self.rms(x, p('input_layernorm'))
            q = (h @ p('self_attn.q_proj').T).view(B, T, self.H, self.hd).transpose(1, 2)
            k = (h @ p('self_attn.k_proj').T).view(B, T, self.KV, self.hd).transpose(1, 2)
            v = (h @ p('self_attn.v_proj').T).view(B, T, self.KV, self.hd).transpose(1, 2)
            q, k = self.rope(q, pos), self.rope(k, pos)
            rep = self.H // self.KV
            k, v = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
            att = (q @ k.transpose(-1, -2)) / (self.hd ** 0.5)
            if bias is not None: att = att + bias
            o = (att.softmax(-1) @ v).transpose(1, 2).reshape(B, T, self.d)
            x = x + o @ p('self_attn.o_proj').T
            h2 = self.rms(x, p('post_attention_layernorm'))
            x = x + (F.silu(h2 @ p('mlp.gate_proj').T) * (h2 @ p('mlp.up_proj').T)) @ p('mlp.down_proj').T
        x = self.rms(x, self.g('model.norm.weight'))
        return x @ self.g('model.embed_tokens.weight').T     # tied head (V+1 classes)
    @torch.no_grad()
    def generate_ar(self, idx, n, temperature=0.9, top_k=40):
        """Plain AR sampling (causal) — used ONLY to capture the donor's own data."""
        for _ in range(n):
            lo = self(idx[:, -1024:])[:, -1, :]
            lo[:, self.V:] = float('-inf')                  # never emit [MASK]
            lo = lo / max(temperature, 1e-6)
            v, _ = torch.topk(lo, top_k)
            lo[lo < v[:, [-1]]] = float('-inf')
            idx = torch.cat([idx, torch.multinomial(lo.softmax(-1), 1)], 1)
        return idx

model = DiffLlama(MODEL_DIR)
MASK_ID = model.V
print(f'loaded: L={model.L} d={model.d} H={model.H}/{model.KV} V={model.V} (+[MASK]={MASK_ID})')
# AR sanity before any adaptation: the donor must speak English
s = model.generate_ar(torch.tensor([[1]], device=DEV), 40, temperature=0.7)
print('AR sample:', repr(decode(s[0, 1:])[:160]))

In [ ]:
# The ONLY data in this port: sequences sampled from the donor itself (the allowed
# 'Q-A captured from the model' budget). Generated BEFORE any weight is touched.
t0 = time.time()
chunks = []
need = N_GEN + N_HELD
gb = 64                                            # generation batch
temps = [0.7, 0.9, 1.0]   # the donor's best behavior; 1.3 degrades a 50M model
with torch.no_grad():
    while sum(c.shape[0] for c in chunks) < need:
        tt = temps[len(chunks) % len(temps)]
        bos = torch.full((gb, 1), 1, dtype=torch.long, device=DEV)
        out = model.generate_ar(bos, SEQ_LEN, temperature=tt)[:, 1:]
        chunks.append(out)
corpus = torch.cat(chunks, 0)[:need]
train_ids, held_ids = corpus[:N_GEN], corpus[N_GEN:]
print(f'corpus: train {tuple(train_ids.shape)}  held {tuple(held_ids.shape)}  '
      f'({train_ids.numel()/1e3:.0f}K train tokens)  in {time.time()-t0:.0f}s')

In [ ]:
# === model-agnostic diffusion core — verbatim from experiments/diffusion_port/diffusion.py
# (verified by 6 CPU unit tests in phase 7). LLaDA ELBO + DiffuLLaMA annealing + Dream sampler.
def sample_mask_rate(batch_size, eps=1e-3, generator=None, device='cpu'):
    u = torch.rand(batch_size, generator=generator, device=device)
    return eps + (1.0 - eps) * u

def anneal_causal_prob(step, total):
    if total <= 0: return 0.0
    return float(max(0.0, min(1.0, 1.0 - min(max(step, 0), total) / total)))

def forward_mask(x0, t, mask_id, generator=None):
    B, L = x0.shape
    noise = torch.rand(B, L, generator=generator, device=x0.device)
    is_masked = noise < t[:, None].expand(B, L)
    empty = ~is_masked.any(dim=1)
    if empty.any():
        force_pos = noise[empty].argmin(dim=1)
        is_masked[empty.nonzero(as_tuple=True)[0], force_pos] = True
    return torch.where(is_masked, torch.full_like(x0, mask_id), x0), is_masked

def diffusion_loss(logits, x0, is_masked, t):
    B, L, V = logits.shape
    ce = F.cross_entropy(logits.reshape(-1, V), x0.reshape(-1), reduction='none').view(B, L)
    return ((ce * is_masked).sum(dim=1) / (t * L)).mean()

def _confidence(probs):
    return probs.max(dim=-1).values                  # maskgit_plus

@torch.no_grad()
def denoise(forward_fn, ids, frozen, steps=64, temperature=0.0, generator=None):
    """Iterative confidence-based unmasking; ids has MASK_ID at unknown positions,
    `frozen` pins known positions (prompt / surviving tokens)."""
    B, L = ids.shape
    for s in range(steps):
        masked = (ids == MASK_ID) & ~frozen
        n_left = int(masked.sum().item())
        if n_left == 0: break
        logits = forward_fn(ids)
        logits[..., MASK_ID:] = float('-inf')         # never emit [MASK]
        if temperature > 0:
            probs = (logits / temperature).softmax(dim=-1)
            pred = torch.multinomial(probs.view(-1, probs.size(-1)), 1, generator=generator).view(B, L)
        else:
            probs = logits.softmax(dim=-1)
            pred = probs.argmax(dim=-1)
        conf = _confidence(probs).masked_fill(~masked, float('-inf'))
        k = min(max(1, n_left // (steps - s)), n_left)
        reveal = conf.view(-1).topk(k).indices
        ids.view(-1)[reveal] = pred.view(-1)[reveal]
    masked = (ids == MASK_ID) & ~frozen
    if masked.any():
        logits = forward_fn(ids); logits[..., MASK_ID:] = float('-inf')
        ids = torch.where(masked, logits.argmax(dim=-1), ids)
    return ids

def diffusion_generate(forward_fn, length, steps=64, prompt_ids=None, temperature=0.0,
                       batch_size=1, generator=None):
    if prompt_ids is not None:
        B, P = prompt_ids.shape
        ids = torch.full((B, length), MASK_ID, dtype=torch.long, device=DEV)
        ids[:, :P] = prompt_ids
        frozen = torch.zeros(B, length, dtype=torch.bool, device=DEV); frozen[:, :P] = True
    else:
        ids = torch.full((batch_size, length), MASK_ID, dtype=torch.long, device=DEV)
        frozen = torch.zeros(batch_size, length, dtype=torch.bool, device=DEV)
    return denoise(forward_fn, ids, frozen, steps, temperature, generator)
print('diffusion core ready')

In [ ]:
@torch.no_grad()
def masked_ce_at(model_, ids, t_val, n=32):
    """Plain mean CE on masked positions at fixed mask rate t (bidirectional forward).
    Comparable to ln(V)=10.37 (uniform) and to the donor's AR next-token CE."""
    x0 = ids[:n]
    t = torch.full((x0.shape[0],), t_val, device=DEV)
    x_t, m = forward_mask(x0, t, MASK_ID)
    lo = model_(x_t, causal=False)
    ce = F.cross_entropy(lo.reshape(-1, lo.shape[-1]), x0.reshape(-1), reduction='none').view(x0.shape)
    return (ce * m).sum().item() / m.sum().item()

@torch.no_grad()
def ar_ce(model_, ids, n=32):
    x = ids[:n]
    lo = model_(x[:, :-1], causal=True)
    return F.cross_entropy(lo.reshape(-1, lo.shape[-1]), x[:, 1:].reshape(-1)).item()

print(f'uniform baseline ln(V)      : {math.log(model.V):.2f} nats')
print(f'donor AR next-token CE (held): {ar_ce(model, held_ids):.2f} nats  (how predictable the corpus is)')
for tv in (0.3, 0.5):
    print(f'PRE-adaptation masked-CE @t={tv}: {masked_ce_at(model, held_ids, tv):.2f} nats  (raw AR weights, bidir)')

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.0)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / WARMUP))
anneal_total = int(STEPS * ANNEAL_FRAC)
g = torch.Generator(device=DEV).manual_seed(7)
ema, t0 = None, time.time()
model.train()
for step in range(1, STEPS + 1):
    bi = torch.randint(0, train_ids.shape[0], (BS,), device=DEV)
    x0 = train_ids[bi]
    rho = anneal_causal_prob(step, anneal_total)
    causal_rows = (torch.rand(BS, device=DEV) < rho)
    t = sample_mask_rate(BS, eps=EPS_T, device=DEV)
    x_t, m = forward_mask(x0, t, MASK_ID)
    loss = diffusion_loss(model(x_t, causal_rows=causal_rows), x0, m, t)
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step()
    ema = loss.item() if ema is None else 0.98 * ema + 0.02 * loss.item()
    if step % 500 == 0 or step == 1:
        model.eval()
        h = masked_ce_at(model, held_ids, 0.5)
        model.train()
        print(f'step {step:>5}  rho {rho:.2f}  train-ELBO(ema) {ema:6.3f}  held masked-CE@0.5 {h:5.2f}  '
              f'({time.time()-t0:.0f}s)')
model.eval()
print('adaptation done')

In [ ]:
# 1) masked-CE across noise levels (held, bidirectional)
print('held masked-CE by mask rate (uniform=10.37, donor-AR-CE see above):')
for tv in (0.3, 0.5, 0.7, 0.9):
    print(f'  t={tv}: {masked_ce_at(model, held_ids, tv):5.2f} nats')

# 2) reconstruction: corrupt 25% of held text, denoise in 16 parallel steps
x0 = held_ids[:8]
t = torch.full((x0.shape[0],), 0.25, device=DEV)
x_c, m = forward_mask(x0, t, MASK_ID)
rec = denoise(lambda i: model(i, causal=False), x_c.clone(), ~m, steps=16)
acc = ((rec == x0) & m).sum().item() / m.sum().item()
print(f'\nreconstruction of 25%-masked held text, 16 steps: token accuracy {acc:.1%} '
      f'(chance ~{1/model.V:.4%})')
print('  original :', repr(decode(x0[0, :48])))
print('  recovered:', repr(decode(rec[0, :48])))

# 3) parallel generation from scratch (the Mercury mode: no autoregression)
gen = diffusion_generate(lambda i: model(i, causal=False), length=128, steps=64,
                         temperature=0.7, batch_size=2, generator=g)
for b in range(gen.shape[0]):
    print(f'\nparallel-denoised sample {b}:', repr(decode(gen[b])[:300]))

# 4) prompt-conditioned parallel infill/continuation
prompt = held_ids[1, :24][None]
cont = diffusion_generate(lambda i: model(i, causal=False), length=96, steps=48,
                          prompt_ids=prompt, temperature=0.7, generator=g)
print('\nprompt    :', repr(decode(prompt[0])))
print('continued :', repr(decode(cont[0, 24:])))

In [ ]:
out_sd = {}
for k in model.names:
    out_sd[k] = model.g(k).detach().cpu().contiguous()
path = '/kaggle/working/supra_mercury2.safetensors' if os.path.isdir('/kaggle/working') \
       else 'supra_mercury2.safetensors'
save_file(out_sd, path)
cfg = json.load(open(os.path.join(MODEL_DIR, 'config.json')))
cfg['vocab_size'] = model.V + 1
cfg['mercury_port'] = {'mask_token_id': MASK_ID, 'attention': 'bidirectional',
                       'objective': 'masked-diffusion (LLaDA ELBO)',
                       'sampler': 'confidence parallel denoising',
                       'data': 'self-generated from donor only'}
json.dump(cfg, open(path.replace('.safetensors', '.config.json'), 'w'), indent=1)
print('saved translated model ->', path, f'({os.path.getsize(path)/1e6:.0f} MB)')

## How to read

- **PRE-adaptation masked-CE** (raw AR weights run bidirectionally) is near-garbage — the
  donor cannot denoise. After adaptation, **held masked-CE** at low mask rates should
  approach the donor's own AR next-token CE (the amount of predictability that exists in
  its text), and stay far below the uniform 10.37 at high rates — proof the backbone now
  *uses bidirectional context*.
- **Reconstruction accuracy** ≫ chance and readable recovered text = the denoiser works.
- **Parallel samples** are generated in 64 steps for 128 tokens (vs 128 AR steps) — the
  Mercury generation mode; coherent English here means the knowledge survived the port.
- The saved `supra_mercury2.safetensors` (+config) is the translated model.

### Honest caveats
- The data budget is tiny vs Dream/LLaDA (0.4M-token corpus, ~32M tokens of training):
  expect locally-coherent text, not donor-equal quality. The point is the **method**:
  knowledge moved across architectures with zero external data.
- Self-generated corpus = the donor's typical set; masked-CE numbers are relative to that
  distribution (that is exactly the 'Q-A captured from the model' framing).
- ELBO and AR perplexity are not directly comparable; we report plain masked-CE for an
  apples-to-apples nats scale.